# What is on the Google Drive?

Lists the shared Drive folder (no login, no download of big files), checks which sessions have what the training needs,
reads the small metadata files, and opens the checkpoints. Writes `drive_index.json` (path, Drive id, size of every file),
which the training notebook can use to fetch only the files it needs.

Runs on Kaggle (Internet on, no GPU needed) or locally. Takes ~5 min, mostly listing ~9,700 items.

In [ ]:
DRIVE_URL = "https://drive.google.com/drive/folders/19yETUSzehXSXpiADBFPc1n9XN8-x41jp"
OPEN_CHECKPOINTS = True   # downloads the .pth files (~270 MB) to print what is inside

In [ ]:
import collections, html, json, os, re, time
from concurrent.futures import ThreadPoolExecutor
import requests

S = requests.Session()
ROOT = re.search(r"folders/([\w-]+)", DRIVE_URL).group(1)

def list_folder(fid):
    # Drive's embedded folder view: lists every entry (gdown stops at 50 per folder)
    t = S.get(f"https://drive.google.com/embeddedfolderview?id={fid}", timeout=60).text
    out = []
    for chunk in t.split('class="flip-entry"')[1:]:
        eid = re.search(r'id="entry-([\w-]+)"', chunk).group(1)
        name = html.unescape(re.search(r'flip-entry-title">([^<]*)', chunk).group(1))
        out.append((name, eid, "/drive/folders/" in chunk))
    return out

def walk(fid, path=""):
    items = []
    for name, eid, folder in list_folder(fid):
        p = f"{path}/{name}" if path else name
        items.append({"path": p, "id": eid, "folder": folder})
        if folder:
            items += walk(eid, p)
    return items

def fetch(fid):
    return S.get(f"https://drive.usercontent.google.com/download?id={fid}&export=download", timeout=600)

def size(fid):
    # bytes without downloading; big files report it on Drive's "too large to scan" page
    r = S.get(f"https://drive.usercontent.google.com/download?id={fid}&export=download", stream=True, timeout=60)
    try:
        if "text/html" not in r.headers.get("Content-Type", ""):
            return int(r.headers.get("Content-Length") or len(r.content))
        m = re.search(r"\(([\d.]+)\s*([KMGT])B?\)", r.text[:50000])
        return int(float(m.group(1)) * {"K": 1e3, "M": 1e6, "G": 1e9, "T": 1e12}[m.group(2)]) if m else None
    finally:
        r.close()

gb = lambda b: f"{b / 1e9:.2f} GB" if b else "?"

## Everything in the folder

In [ ]:
t = time.time()
items = walk(ROOT)
by = {i["path"]: i for i in items}
files = [i for i in items if not i["folder"]]
print(f"{len(items)} items, {len(files)} files, listed in {time.time() - t:.0f} s\n")
for top in sorted({i["path"].split("/")[0] for i in items}):
    sub = [i for i in files if i["path"].startswith(top + "/")]
    ext = collections.Counter(os.path.splitext(i["path"])[1] or "(none)" for i in sub)
    print(f"{top + '/':<14}{len(sub):>6} files  {dict(ext.most_common(6))}")
empty = [i["path"] for i in items if i["folder"] and not any(j["path"].startswith(i["path"] + "/") for j in items)]
print(f"\n{len(empty)} empty folders, e.g. {empty[:8]}")

## Sizes of the non-per-frame files

In [ ]:
big = [i for i in files if not re.search(r"/(left|right|leap)/frame_\d+\.(png|json)$", i["path"])]
with ThreadPoolExecutor(8) as ex:
    for i, s in zip(big, ex.map(lambda i: size(i["id"]), big)):
        i["size"] = s
for top in sorted({i["path"].split("/")[0] for i in big}):
    sub = [i for i in big if i["path"].startswith(top + "/")]
    print(f"{top + '/':<14}{len(sub):>5} files {gb(sum(i['size'] or 0 for i in sub)):>10}   largest:",
          [(os.path.basename(i["path"]), gb(i["size"])) for i in sorted(sub, key=lambda i: -(i["size"] or 0))[:3]])
pngs = [i for i in files if i["path"].endswith(".png")]
if pngs:
    s = size(pngs[0]["id"])
    print(f"\n{len(pngs)} per-frame PNGs, ~{s / 1e3:.0f} KB each -> ~{gb(s * len(pngs))}")
json.dump(items, open("drive_index.json", "w"), indent=0)
print("wrote drive_index.json")

## Two-camera sessions (this repo's format, `sessions/`)

In [ ]:
NEED = ["session.json", "groups.json", "cam_A/frames.bin", "cam_A/frames.csv", "cam_B/frames.bin", "cam_B/frames.csv",
        "mano/smoothed.npz"]
sess = sorted({m.group(1) for i in items if (m := re.match(r"(sessions/[^/]+)/", i["path"] + "/"))})
print(f"{'session':<30}" + "".join(f"{n.split('/')[-1][:9]:>11}" for n in NEED) + "   notes")
for s in sess:
    have = ["yes" if f"{s}/{n}" in by else "-" for n in NEED]
    notes = ""
    if f"{s}/session.json" in by:
        try:
            notes = fetch(by[f"{s}/session.json"]["id"]).json().get("notes", "")
        except Exception as e:
            notes = f"(unreadable: {e})"
    print(f"{s:<30}" + "".join(f"{h:>11}" for h in have) + f"   {notes[:50]}")
ok = [s for s in sess if all(f"{s}/{n}" in by for n in NEED)]
print(f"\n{len(ok)} of {len(sess)} sessions have everything the two-camera training needs")

## Single-camera dataset (`dataset/`)

In [ ]:
ds = sorted({i["path"].split("/")[1] for i in items if re.match(r"dataset/session_", i["path"])})
val = set(fetch(by["dataset/val_sessions.txt"]["id"]).text.split()) if "dataset/val_sessions.txt" in by else set()
excl = fetch(by["dataset/excluded_frames.txt"]["id"]).text.split() if "dataset/excluded_frames.txt" in by else []
count = lambda s, d: sum(1 for i in files if i["path"].startswith(f"dataset/{s}/{d}/"))

def meta(s):
    p = f"dataset/{s}/metadata.json"
    return fetch(by[p]["id"]).json() if p in by else {}

with ThreadPoolExecutor(8) as ex:
    metas = list(ex.map(meta, ds))
rows = []
print(f"{'session':<26}{'left':>6}{'right':>6}{'leap':>6}  {'pose_label':<16}{'save_fps':>9}  device  split")
for s, m in zip(ds, metas):
    l, r, j = count(s, "left"), count(s, "right"), count(s, "leap")
    rows.append((s, m.get("pose_label", "?"), min(l, r, j)))
    print(f"{s:<26}{l:>6}{r:>6}{j:>6}  {m.get('pose_label', '?'):<16}{m.get('save_fps', '?'):>9}  "
          f"{m.get('device', '?')[-30:-20]}  {'val' if s in val else 'train'}")
poses = collections.Counter()
for s, p, n in rows:
    poses[p] += n
print(f"\nframes per pose label: {dict(poses)}")
print(f"{len(ds)} sessions, {sum(n for *_, n in rows)} stereo frames with joints, {len(val)} val sessions, "
      f"{len(excl)} excluded entries")

In [ ]:
# one per-frame label and one calibration, to see the format
s = next(s for s in ds if count(s, "leap"))
p = sorted(i["path"] for i in files if i["path"].startswith(f"dataset/{s}/leap/"))[0]
lab = fetch(by[p]["id"]).json()
print(p, "\n", {k: v for k, v in lab.items() if k != "joints"})
print("joints:", list(lab.get("joints", {}))[:25])
c = next((i for i in files if i["path"].endswith("calibration.json")), None)
if c:
    print("\n", c["path"], "keys:", {k: list(v) if isinstance(v, dict) else v for k, v in fetch(c["id"]).json().items()})

## Checkpoints

In [ ]:
if OPEN_CHECKPOINTS:
    import io, torch
    for i in [i for i in files if i["path"].startswith("checkpoints/")]:
        r = fetch(i["id"])
        if "text/html" in r.headers.get("Content-Type", ""):  # big file: confirm through Drive's warning page
            form = dict(re.findall(r'name="(\w+)" value="([^"]*)"', r.text))
            r = S.get("https://drive.usercontent.google.com/download", params=form, timeout=600)
        ck = torch.load(io.BytesIO(r.content), map_location="cpu", weights_only=False)
        print(f"\n{i['path']} ({len(r.content) / 1e6:.0f} MB)")
        if isinstance(ck, dict):
            for k, v in ck.items():
                if isinstance(v, dict) and v and all(hasattr(x, "shape") for x in v.values()):
                    ks = list(v)
                    print(f"  {k}: {len(ks)} tensors, first {ks[:3]}, last {[(n, tuple(v[n].shape)) for n in ks[-2:]]}")
                elif hasattr(v, "shape"):
                    print(f"  {k}: tensor {tuple(v.shape)}")
                else:
                    print(f"  {k}: {str(v)[:200]}")